In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings("ignore")


In [2]:
! pip install xgboost lightgbm catboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 4.3 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.7 MB/s  0:00:00
   ---------------------------------------- 0.0/102.4 MB ? eta -:--:--
   ---------------------------------------- 1.0/102.4 MB 4.8 MB/s eta 0:00:22
    --------------------------------------- 1.8/102.4 MB 5.3 MB/s eta 0:00:20
   - -------------------------------------- 2.9/102.4 MB 4.8 MB/s eta 0:00:21
   - -------------------------------------- 4.2/102.4 MB 4.9 MB/s eta 0:00:21
   - -------------------------------------- 5.0/102.4 MB 4.9 MB/s eta 0:00:20
   -- ------------------------------------- 6.0/102.4 MB 5.0 MB/s eta 0:00:20
   -- ------------------------------------- 6.8/102.4 MB 4.9 MB/s eta 0:00:20
   --- ------------------------------------ 7.9/102.4 MB 4.7 MB/s eta 0:00:21
   --- --

In [4]:
# Update these paths if needed
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"
SUB_PATH=  "submission.csv" # template (optional)

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)


try:
    submission = pd.read_csv(SUB_PATH)
    print("Loaded submission template:", submission.shape)
except Exception as e:
    submission = pd.DataFrame({"id": test["id"]})
    print("No template found. Created submission with id only.")
    
train.head()


Train shape: (15000, 10)
Test shape : (10000, 9)
Loaded submission template: (10000, 2)


,id,Sex,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age
0,0,F,1.6000,1.2500,0.5750,43.700754,15.025235,10.262519,13.324265,17.0
1,1,M,1.5750,1.1625,0.3750,32.417653,12.927372,6.208540,8.930093,20.0
2,2,F,1.6125,1.2750,0.4375,34.373769,14.784264,9.142714,10.772810,10.0
3,3,M,1.2250,0.9875,0.3250,18.455524,8.037083,3.926406,4.819415,9.0
4,4,I,1.3750,1.0250,0.3875,22.736299,9.879801,5.513978,6.945627,10.0


In [5]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # One-hot for Sex
    df["Sex_M"] = (df["Sex"] == "M").astype(int)
    df["Sex_F"] = (df["Sex"] == "F").astype(int)
    df["Sex_I"] = (df["Sex"] == "I").astype(int)

    # Volume / Density
    df["Volume"] = df["Length"] * df["Diameter"] * df["Height"]
    df["Density"] = df["Weight"] / (df["Volume"] + 1e-6)

    # Weight ratios
    df["Shucked_Weight_Ratio"] = df["Shucked Weight"] / (df["Weight"] + 1e-6)
    df["Viscera_Weight_Ratio"] = df["Viscera Weight"] / (df["Weight"] + 1e-6)
    df["Shell_Weight_Ratio"] = df["Shell Weight"] / (df["Weight"] + 1e-6)

    # Size / Surface area
    df["Size"] = df["Length"] + df["Diameter"] + df["Height"]
    df["Surface_Area"] = 2 * (
        df["Length"] * df["Diameter"] +
        df["Length"] * df["Height"] +
        df["Diameter"] * df["Height"]
    )

    # Meat features
    df["Meat_Weight"] = df["Shucked Weight"] + df["Viscera Weight"]
    df["Meat_Shell_Ratio"] = df["Meat_Weight"] / (df["Shell Weight"] + 1e-6)

    # Interactions
    df["Length_Diameter"] = df["Length"] * df["Diameter"]
    df["Length_Height"] = df["Length"] * df["Height"]
    df["Diameter_Height"] = df["Diameter"] * df["Height"]

    # Squares
    df["Length_sq"] = df["Length"] ** 2
    df["Diameter_sq"] = df["Diameter"] ** 2
    df["Height_sq"] = df["Height"] ** 2
    df["Weight_sq"] = df["Weight"] ** 2

    # Ratios
    df["Length_Diameter_Ratio"] = df["Length"] / (df["Diameter"] + 1e-6)
    df["Length_Height_Ratio"] = df["Length"] / (df["Height"] + 1e-6)
    df["Diameter_Height_Ratio"] = df["Diameter"] / (df["Height"] + 1e-6)

    # Drop original Sex
    df = df.drop(columns=["Sex"])

    return df


In [6]:
train_fe = create_features(train)
test_fe  = create_features(test)

# NOTE: Your dataset uses 'Age' as target (capital A)
TARGET_COL = "Age"
ID_COL = "id"

if TARGET_COL not in train_fe.columns:
    raise ValueError(f"Target '{TARGET_COL}' not found in train. Available columns: {train_fe.columns.tolist()}")

X = train_fe.drop(columns=[ID_COL, TARGET_COL])
y = train_fe[TARGET_COL].copy()

X_test = test_fe.drop(columns=[ID_COL])

print("Num features:", X.shape[1])
print("First 10 feature names:", list(X.columns)[:10])


Num features: 29
First 10 feature names: ['Length', 'Diameter', 'Height', 'Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight', 'Sex_M', 'Sex_F', 'Sex_I']


In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)


In [8]:
models = {
    "XGBoost": XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1,
        random_state=42,
        n_jobs=-1
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=50,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    ),
    "CatBoost": CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        subsample=0.8,
        random_state=42,
        verbose=0
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=500,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=500,
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        min_samples_split=5,
        min_samples_leaf=2,
        subsample=0.8,
        random_state=42
    )
}


In [9]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

oof_pred_by_model = {}
test_pred_by_model = {}
mae_by_model = {}

print("="*60)
print("Training with 5-Fold CV (MAE)")
print("="*60)

for name, model in models.items():
    print(f"\nModel: {name}")
    oof_pred_model = np.zeros(len(X))
    test_pred_model = np.zeros(len(X_test))
    fold_maes = []

    for fold, (tr_idx, va_idx) in enumerate(kfold.split(X_scaled), 1):
        X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model.fit(X_tr, y_tr)

        va_pred = model.predict(X_va)
        fold_mae = np.mean(np.abs(y_va - va_pred))
        fold_maes.append(fold_mae)

        oof_pred_model[va_idx] = va_pred
        test_pred_model += model.predict(X_test_scaled) / kfold.n_splits

        print(f"  Fold {fold} MAE: {fold_mae:.4f}")

    avg_mae = float(np.mean(fold_maes))
    print(f"  Average MAE: {avg_mae:.4f}")

    oof_pred_by_model[name] = oof_pred_model
    test_pred_by_model[name] = test_pred_model
    mae_by_model[name] = avg_mae


Training with 5-Fold CV (MAE)

Model: XGBoost
  Fold 1 MAE: 1.4317
  Fold 2 MAE: 1.3999
  Fold 3 MAE: 1.4338
  Fold 4 MAE: 1.4626
  Fold 5 MAE: 1.3963
  Average MAE: 1.4248

Model: LightGBM
  Fold 1 MAE: 1.4252
  Fold 2 MAE: 1.3799
  Fold 3 MAE: 1.4051
  Fold 4 MAE: 1.4443
  Fold 5 MAE: 1.3894
  Average MAE: 1.4088

Model: CatBoost
  Fold 1 MAE: 1.3804
  Fold 2 MAE: 1.3430
  Fold 3 MAE: 1.3703
  Fold 4 MAE: 1.4158
  Fold 5 MAE: 1.3303
  Average MAE: 1.3679

Model: RandomForest
  Fold 1 MAE: 1.3939
  Fold 2 MAE: 1.3518
  Fold 3 MAE: 1.3790
  Fold 4 MAE: 1.4125
  Fold 5 MAE: 1.3430
  Average MAE: 1.3761

Model: ExtraTrees
  Fold 1 MAE: 1.3994
  Fold 2 MAE: 1.3631
  Fold 3 MAE: 1.3713
  Fold 4 MAE: 1.4190
  Fold 5 MAE: 1.3446
  Average MAE: 1.3795

Model: GradientBoosting
  Fold 1 MAE: 1.4122
  Fold 2 MAE: 1.3804
  Fold 3 MAE: 1.4032
  Fold 4 MAE: 1.4366
  Fold 5 MAE: 1.3764
  Average MAE: 1.4017


In [10]:
# Weight = inverse MAE
raw_weights = {name: 1.0 / mae for name, mae in mae_by_model.items()}
total = sum(raw_weights.values())
weights = {name: w / total for name, w in raw_weights.items()}

print("\n" + "="*60)
print("Model Weights (normalized)")
print("="*60)
for name, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"{name:16s}: {w:.4f}")

# Weighted OOF + Weighted Test
oof_ensemble = np.zeros(len(X))
test_ensemble = np.zeros(len(X_test))

for name in models.keys():
    oof_ensemble += oof_pred_by_model[name] * weights[name]
    test_ensemble += test_pred_by_model[name] * weights[name]

overall_mae = float(np.mean(np.abs(y - oof_ensemble)))
print("\nOverall Out-of-Fold MAE:", round(overall_mae, 4))



Model Weights (normalized)
CatBoost        : 0.1697
RandomForest    : 0.1687
ExtraTrees      : 0.1683
GradientBoosting: 0.1656
LightGBM        : 0.1648
XGBoost         : 0.1629

Overall Out-of-Fold MAE: 1.3716


In [12]:
# Ensure correct columns
if "id" in submission.columns:
    submission["Age"] = test_ensemble
else:
    submission = pd.DataFrame({"id": test["id"], "Age": test_ensemble})

OUT_PATH ="submission1.csv"
submission.to_csv(OUT_PATH, index=False)

print("Saved submission:", OUT_PATH)
print("Predicted Age range:", (float(test_ensemble.min()), float(test_ensemble.max())))

print("\nTarget MAE: < 1.25")
print("Achieved MAE:", round(overall_mae, 4))
print("SUCCESS ✅" if overall_mae < 1.25 else f"Not yet ❌ (gap: {overall_mae - 1.25:.4f})")

submission.head()


Saved submission: submission1.csv
Predicted Age range: (3.6944789474593764, 19.599303791404562)

Target MAE: < 1.25
Achieved MAE: 1.3716
Not yet ❌ (gap: 0.1216)


,id,Age
0,15000,14.463849
1,15001,13.103299
2,15002,9.509111
3,15003,11.058606
4,15004,9.288591
